# Ensemble Methods Lab

From decision trees to boosting - understanding how combining weak learners creates strong models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_wine, make_classification
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, 
                              GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     learning_curve, GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import accuracy_score
import time
np.random.seed(42)
%matplotlib inline

## 1. Decision Tree Visualization

In [ ]:
# Simple decision tree on iris
iris = load_iris()
X_iris, y_iris = iris.data[:, :2], iris.target  # use 2 features for viz

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_iris, y_iris)

print(export_text(tree, feature_names=iris.feature_names[:2]))
print(f'\nAccuracy: {tree.score(X_iris, y_iris):.3f}')

In [ ]:
# Visualize decision boundary
def plot_decision_boundary(model, X, y, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='k', s=20)
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for depth, ax in zip([1, 3, None], axes):
    t = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_iris, y_iris)
    plot_decision_boundary(t, X_iris, y_iris, ax, f'Depth={depth}, Acc={t.score(X_iris, y_iris):.3f}')
plt.tight_layout()
plt.show()

## 2. Bagging: Variance Reduction

In [ ]:
# Show how bagging reduces variance
X, y = make_classification(n_samples=500, n_features=20, n_informative=10, 
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Single tree vs Bagging with different n_estimators
n_estimators_range = [1, 5, 10, 25, 50, 100]
train_scores = []
test_scores = []

for n in n_estimators_range:
    bag = BaggingClassifier(n_estimators=n, random_state=42)
    bag.fit(X_train, y_train)
    train_scores.append(bag.score(X_train, y_train))
    test_scores.append(bag.score(X_test, y_test))

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, train_scores, 'b-o', label='Train')
plt.plot(n_estimators_range, test_scores, 'r-o', label='Test')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Bagging: Variance Reduction with More Trees')
plt.legend()
plt.grid(True)
plt.show()
print(f'Single tree test acc: {test_scores[0]:.3f}')
print(f'100 trees test acc: {test_scores[-1]:.3f}')

## 3. Random Forest: Feature Importance

In [ ]:
# Random Forest on wine dataset
wine = load_wine()
X_wine, y_wine = wine.data, wine.target
X_tr, X_te, y_tr, y_te = train_test_split(X_wine, y_wine, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f'Random Forest Accuracy: {rf.score(X_te, y_te):.3f}')

# Feature importance
importances = rf.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[idx])
plt.xticks(range(len(importances)), [wine.feature_names[i] for i in idx], rotation=45, ha='right')
plt.title('Random Forest Feature Importance (Wine Dataset)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 4. Gradient Boosting from Scratch

In [ ]:
# Simplified Gradient Boosting for regression (concept)
from sklearn.tree import DecisionTreeRegressor

# Generate simple regression data
np.random.seed(42)
X_gb = np.sort(np.random.uniform(0, 10, 200)).reshape(-1, 1)
y_gb = np.sin(X_gb.ravel()) + np.random.normal(0, 0.2, 200)

# Gradient Boosting from scratch
class SimpleGradientBoosting:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.predictions_history = []
    
    def fit(self, X, y):
        # Initialize with mean
        self.initial_pred = y.mean()
        current_pred = np.full(len(y), self.initial_pred)
        self.predictions_history.append(current_pred.copy())
        
        for i in range(self.n_estimators):
            # Compute residuals (negative gradient for MSE)
            residuals = y - current_pred
            
            # Fit tree to residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            # Update predictions
            current_pred += self.learning_rate * tree.predict(X)
            self.predictions_history.append(current_pred.copy())
        
        return self
    
    def predict(self, X):
        pred = np.full(X.shape[0], self.initial_pred)
        for tree in self.trees:
            pred += self.learning_rate * tree.predict(X)
        return pred

gb = SimpleGradientBoosting(n_estimators=20, learning_rate=0.3, max_depth=2)
gb.fit(X_gb, y_gb)
print(f'Built {len(gb.trees)} trees sequentially fitting residuals')

In [ ]:
# Visualize how gradient boosting builds up predictions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
stages = [0, 1, 3, 5, 10, 20]
for ax, stage in zip(axes.ravel(), stages):
    ax.scatter(X_gb, y_gb, alpha=0.3, s=10, label='Data')
    ax.plot(X_gb, gb.predictions_history[stage], 'r-', linewidth=2, label=f'Prediction')
    ax.set_title(f'After {stage} trees')
    ax.legend()
    ax.set_ylim(-2, 2)
plt.suptitle('Gradient Boosting: Sequential Residual Fitting', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Compare All Ensemble Methods

In [ ]:
# Compare methods on classification
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    elapsed = time.time() - start
    results[name] = {'mean_acc': scores.mean(), 'std_acc': scores.std(), 'time': elapsed}
    print(f'{name:20s}: {scores.mean():.4f} (+/- {scores.std():.4f}) [{elapsed:.3f}s]')

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
means = [r['mean_acc'] for r in results.values()]
stds = [r['std_acc'] for r in results.values()]
ax.barh(names, means, xerr=stds, capsize=5)
ax.set_xlabel('Accuracy')
ax.set_title('Ensemble Methods Comparison')
ax.set_xlim(0.8, 1.0)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning

In [ ]:
# Effect of key hyperparameters
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# max_depth effect
depths = range(1, 20)
train_s, test_s = [], []
for d in depths:
    rf_d = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
    rf_d.fit(X_train, y_train)
    train_s.append(rf_d.score(X_train, y_train))
    test_s.append(rf_d.score(X_test, y_test))
axes[0].plot(depths, train_s, 'b-', label='Train')
axes[0].plot(depths, test_s, 'r-', label='Test')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Effect of max_depth')
axes[0].legend()

# n_estimators effect
n_est = [5, 10, 25, 50, 100, 200]
train_s, test_s = [], []
for n in n_est:
    rf_n = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_n.fit(X_train, y_train)
    train_s.append(rf_n.score(X_train, y_train))
    test_s.append(rf_n.score(X_test, y_test))
axes[1].plot(n_est, train_s, 'b-o', label='Train')
axes[1].plot(n_est, test_s, 'r-o', label='Test')
axes[1].set_xlabel('n_estimators')
axes[1].set_title('Effect of n_estimators')
axes[1].legend()

# learning_rate effect (GBM)
lrs = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
train_s, test_s = [], []
for lr in lrs:
    gb_lr = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=42)
    gb_lr.fit(X_train, y_train)
    train_s.append(gb_lr.score(X_train, y_train))
    test_s.append(gb_lr.score(X_test, y_test))
axes[2].plot(lrs, train_s, 'b-o', label='Train')
axes[2].plot(lrs, test_s, 'r-o', label='Test')
axes[2].set_xlabel('learning_rate')
axes[2].set_title('Effect of learning_rate (GBM)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. Learning Curves for Overfitting Diagnosis

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_lc = [
    ('Decision Tree (no limit)', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=50, random_state=42)),
]

for ax, (name, model) in zip(axes, models_lc):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy', n_jobs=-1)
    
    ax.plot(train_sizes, train_scores.mean(axis=1), 'b-o', label='Train', markersize=4)
    ax.plot(train_sizes, val_scores.mean(axis=1), 'r-o', label='Validation', markersize=4)
    ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='red')
    ax.set_xlabel('Training Size')
    ax.set_ylabel('Accuracy')
    ax.set_title(name)
    ax.legend()
    ax.set_ylim(0.7, 1.05)

plt.tight_layout()
plt.show()
print('Gap between train and val = overfitting')
print('Both low = underfitting')

## 8. Feature Importance Comparison

In [ ]:
# Compare feature importances across methods
rf2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
gb2 = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
ada2 = AdaBoostClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, model, name in zip(axes, [rf2, gb2, ada2], ['Random Forest', 'Gradient Boosting', 'AdaBoost']):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:10]  # top 10
    ax.barh(range(10), imp[idx])
    ax.set_yticks(range(10))
    ax.set_yticklabels([wine.feature_names[i] for i in idx])
    ax.set_title(f'{name} Feature Importance')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

## Summary

| Method | Key Idea | Bias | Variance | Speed |
|--------|----------|------|----------|-------|
| Decision Tree | Single tree | Low | High | Fast |
| Bagging/RF | Average many trees | Low | Reduced | Medium |
| AdaBoost | Weight hard examples | Reduced | Medium | Medium |
| Gradient Boosting | Fit residuals | Reduced | Low-Med | Slower |